In [ ]:
# Setup the Dash version
from dash import Dash, dcc, html, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl
import plotly.express as px
import pandas as pd
import base64
from enhanced_animal_shelter import AnimalShelter



In [ ]:
# Initialize shelter with role simulation
db = AnimalShelter("aacuser", "SNHU1234", role="admin")  # Change to "user" to test restrictions

import pandas as pd
from enhanced_animal_shelter import AnimalShelter

# Read CSV
csv_file = "aac_shelter_outcomes.csv"  # Change to your actual CSV filename
df_csv = pd.read_csv(csv_file)

# Insert each row as a document
inserted_count = 0
for _, row in df_csv.iterrows():
    doc = row.to_dict()
    # Convert any NaN to None (MongoDB friendly)
    doc = {k: None if pd.isna(v) else v for k, v in doc.items()}
    db.create(doc)
    inserted_count += 1

print(f"Successfully inserted {inserted_count} records from CSV into MongoDB")

In [ ]:
df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True) if '_id' in df.columns else None

In [ ]:
app = Dash(__name__)

image_filename = 'logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))),
    html.Center(html.H1('Grazioso Salvare Dashboard - All Animals')),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        row_selectable="single",
        selected_rows=[0],
        page_size=10,
        sort_action="native",
        style_table={'overflowX': 'auto'},
    ),
    html.Br(),
    html.Hr(),
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ]),
    html.H3("Dashboard by Amaro Terrazas")
])

In [ ]:
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    if filter_type == 'Reset':
        df = pd.DataFrame.from_records(db.read({}))
    elif filter_type == 'Water':
        df = pd.DataFrame.from_records(db.get_rescue_stats('Water'))
    elif filter_type == 'Mountain':
        df = pd.DataFrame.from_records(db.get_rescue_stats('Mountain'))
    elif filter_type == 'Disaster':
        df = pd.DataFrame.from_records(db.get_rescue_stats('Disaster'))
    else:
        df = pd.DataFrame.from_records(db.read({}))

    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    # If aggregated data, rename columns for consistency with raw data
    if 'breed' not in df.columns and '_id' in df.columns:
        df.rename(columns={'_id': 'breed', 'count': 'breed_count'}, inplace=True)

    return df.to_dict('records')

In [ ]:
@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data")
)
def update_graphs(viewData):
    if viewData is None or not viewData:
        return [html.P("No data available for chart.")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    if 'breed' in dff.columns:
        fig = px.pie(dff, names='breed', title='Breed Distribution (All Loaded Animals)')
        return [dcc.Graph(figure=fig)]
    
    return [html.P("No breed column found in data.")]

In [ ]:
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, selected_rows):
    if viewData is None or not viewData:
        return [html.P("No data available for mapping.")]
    
    dff = pd.DataFrame.from_dict(viewData)
    row = selected_rows[0] if selected_rows else 0
    
    try:
        lat = float(dff.iloc[row]['location_lat']) if 'location_lat' in dff.columns else 30.2672
        lon = float(dff.iloc[row]['location_long']) if 'location_long' in dff.columns else -97.7431
        breed = dff.iloc[row]['breed'] if 'breed' in dff.columns else "Unknown"
        name = dff.iloc[row]['name'] if 'name' in dff.columns else "Unknown"
    except:
        lat, lon = 30.2672, -97.7431
        breed, name = "Unknown", "Unknown"
    
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(position=[lat, lon], children=[
                    dl.Tooltip(breed),
                    dl.Popup([html.H1("Animal Name"), html.P(name)])
                ])
            ]
        )
    ]

In [ ]:
app.run(mode='inline', port=8050, debug=False)